## 📚 Diccionario de Datos

| # | Variable | Tipo | Descripción |
|---|----------|------|-------------|
| 1 | `Hours_Studied` | Numérica | Horas de estudio por semana |
| 2 | `Attendance` | Numérica | Porcentaje de asistencia a clases |
| 3 | `Parental_Involvement` | Categórica | Nivel de involucramiento de los padres (`Low`, `Medium`, `High`) |
| 4 | `Access_to_Resources` | Categórica | Acceso a recursos educativos (`Low`, `Medium`, `High`) |
| 5 | `Extracurricular_Activities` | Categórica | Participación en actividades extracurriculares (`Yes`, `No`) |
| 6 | `Sleep_Hours` | Numérica | Promedio de horas de sueño por noche |
| 7 | `Previous_Scores` | Numérica | Puntaje obtenido en exámenes anteriores |
| 8 | `Motivation_Level` | Categórica | Nivel de motivación del estudiante (`Low`, `Medium`, `High`) |
| 9 | `Internet_Access` | Categórica | Acceso a internet (`Yes`, `No`) |
| 10 | `Tutoring_Sessions` | Numérica | Número de sesiones de tutoría por mes |
| 11 | `Family_Income` | Categórica | Nivel de ingresos familiares (`Low`, `Medium`, `High`) |
| 12 | `Teacher_Quality` | Categórica | Calidad del docente (`Low`, `Medium`, `High`) |
| 13 | `School_Type` | Categórica | Tipo de institución educativa (`Public`, `Private`) |
| 14 | `Peer_Influence` | Categórica | Influencia del entorno social (`Positive`, `Neutral`, `Negative`) |
| 15 | `Physical_Activity` | Numérica | Horas de actividad física por semana |
| 16 | `Learning_Disabilities` | Categórica | Presencia de discapacidades de aprendizaje (`Yes`, `No`) |
| 17 | `Parental_Education_Level` | Categórica | Nivel educativo de los padres (`High School`, `College`, `Postgraduate`) |
| 18 | `Distance_from_Home` | Categórica | Distancia del hogar a la escuela (`Near`, `Moderate`, `Far`) |
| 19 | `Gender` | Categórica | Género del estudiante (`Male`, `Female`) |
| 20 | `Exam_Score` | Numérica | 🎯 **Variable objetivo** — Puntaje obtenido en el examen final |

## 🔍 Diferencias entre `ROW_NUMBER()`, `RANK()` y `DENSE_RANK()`

| Función | ¿Qué hace? | Empates (`ties`) | ¿Deja huecos en el ranking? | Caso típico de uso |
|---|---|---|---|---|
| `ROW_NUMBER()` | Asigna un número único y secuencial a cada fila | ❌ No respeta empates (siempre números distintos) | ❌ No | Cuando necesitas un identificador único por orden (Top 1 exacto, paginación, deduplicación) |
| `RANK()` | Asigna el mismo rango a filas empatadas | ✅ Sí | ✅ Sí | Rankings competitivos (si hay empate en 1°, el siguiente es 3°) |
| `DENSE_RANK()` | Asigna el mismo rango a filas empatadas | ✅ Sí | ❌ No | Rankings “compactos” sin saltos (si hay empate en 1°, el siguiente es 2°) |

### 🧠 Ejemplo rápido (puntajes: `100, 95, 95, 90`)

| Puntaje | `ROW_NUMBER()` | `RANK()` | `DENSE_RANK()` |
|---|---:|---:|---:|
| 100 | 1 | 1 | 1 |
| 95  | 2 | 2 | 2 |
| 95  | 3 | 2 | 2 |
| 90  | 4 | 4 | 3 |

In [2]:
import duckdb
import os
os.listdir('../data/student_performance')
con = duckdb.connect()
con.execute("CREATE TABLE student_performance AS SELECT * FROM read_csv_auto('../data/student_performance/student_performance.csv')")
result = con.execute("SELECT * FROM student_performance").fetchdf()
print(result.head())


   Hours_Studied  Attendance Parental_Involvement Access_to_Resources  \
0             23          84                  Low                High   
1             19          64                  Low              Medium   
2             24          98               Medium              Medium   
3             29          89                  Low              Medium   
4             19          92               Medium              Medium   

   Extracurricular_Activities  Sleep_Hours  Previous_Scores Motivation_Level  \
0                       False            7               73              Low   
1                       False            8               59              Low   
2                        True            7               91           Medium   
3                        True            8               98           Medium   
4                        True            6               65           Medium   

   Internet_Access  Tutoring_Sessions Family_Income Teacher_Quality  \
0        

## 🥇 1️⃣ ROW_NUMBER()

> Asigna un número único a cada fila.

📌 No le importa si hay empates.

Si dos estudiantes tienen el mismo puntaje, igual les da números distintos.

### 1) 🏆 Identificación única por rendimiento
**Objetivo:**  
Generar una lista de todos los estudiantes numerados del **1 al N**, ordenados por `Exam_Score` de **mayor a menor**.


---


In [6]:
con.execute("""
-- ¿Para qué sirve? Crear un ID de posición único para procesar
-- registros uno por uno en un pipeline (ej. envío de reportes por lote)
SELECT
    exam_score,
    ROW_NUMBER() OVER (ORDER BY exam_score DESC) AS rn_global
FROM student_performance
ORDER BY rn_global;
""").fetch_df()

,Exam_Score,rn_global
0,101,1
1,100,2
2,99,3
3,99,4
4,98,5
...,...,...
6602,57,6603
6603,57,6604
6604,57,6605
6605,56,6606



### 2) 🏫 Top estudiantes por tipo de escuela
**Objetivo:**  
Numerar a los estudiantes dentro de cada `School_Type` (`Public` o `Private`), ordenándolos por `Hours_Studied` en forma **descendente**.





In [15]:
con.execute("""
-- ¿Para qué sirve? Extraer el "alumno #1 que más estudia"
-- de cada tipo de institución para comparativas de gestión
SELECT
    school_type,
    hours_studied,
    ROW_NUMBER() OVER (
        PARTITION BY school_type
        ORDER BY hours_studied DESC
    ) AS rn_per_school
FROM student_performance
ORDER BY school_type, rn_per_school;
""").fetch_df()

,School_Type,Hours_Studied,rn_per_school
0,Private,44,1
1,Private,39,2
2,Private,39,3
3,Private,39,4
4,Private,38,5
...,...,...,...
6602,Public,2,4594
6603,Public,2,4595
6604,Public,2,4596
6605,Public,1,4597


---

### 3) 👩‍🎓👨‍🎓 Mayor asistencia por género
**Objetivo:**  
Dentro de cada grupo de `Gender`, numerar a los alumnos según su `Attendance` (de mayor a menor).


---


In [19]:
con.execute("""
-- ¿Para qué sirve? Segmentar para campañas de incentivos
-- personalizadas garantizando orden determinístico por grupo
SELECT

    gender,
    attendance,
    ROW_NUMBER() OVER (
        PARTITION BY gender
        ORDER BY attendance DESC
    ) AS rn_per_gender
FROM student_performance
ORDER BY gender, rn_per_gender;
""").fetch_df()

,Gender,Attendance,rn_per_gender
0,Female,100,1
1,Female,100,2
2,Female,100,3
3,Female,100,4
4,Female,100,5
...,...,...,...
6602,Male,60,3810
6603,Male,60,3811
6604,Male,60,3812
6605,Male,60,3813


## 🥈 2️⃣ `RANK()`

> A diferencia de `ROW_NUMBER()`, `RANK()` asigna el mismo rango a valores idénticos y deja un **hueco** en la secuencia posterior.  
> Esto es clave para análisis competitivos y criterios de justicia.

---



### 1) 🏆 Competencia por Excelencia Académica
**Objetivo:**  
Clasificar a los estudiantes según `Exam_Score`. Si dos estudiantes tienen el mismo puntaje, deben compartir el mismo rango.



In [23]:
con.execute("""
-- ¿Para qué sirve? Asignación de becas justa: si dos alumnos
-- empatan en posición 1, el siguiente recibe la posición 3
SELECT
    exam_score,
    RANK() OVER (ORDER BY exam_score DESC) AS rank_exam
FROM student_performance
ORDER BY rank_exam;
""").fetch_df()

,Exam_Score,rank_exam
0,101,1
1,100,2
2,99,3
3,99,3
4,98,5
...,...,...
6602,57,6602
6603,57,6602
6604,57,6602
6605,56,6606


---

### 2) 👨‍👩‍👧‍👦 Liderazgo por Nivel de Involucramiento
**Objetivo:**  
Crear un ranking de `Attendance` particionado por `Parental_Involvement`.

---




In [26]:
con.execute("""
-- ¿Para qué sirve? Medir si el apoyo parental determina
-- quiénes son los líderes de asistencia en cada categoría
SELECT
    parental_involvement,
    attendance,
    RANK() OVER (
        PARTITION BY parental_involvement
        ORDER BY attendance DESC
    ) AS rank_attendance
FROM student_performance
ORDER BY parental_involvement, rank_attendance;
""").fetch_df()


,Parental_Involvement,Attendance,rank_attendance
0,High,100,1
1,High,100,1
2,High,100,1
3,High,100,1
4,High,100,1
...,...,...,...
6602,Medium,60,3312
6603,Medium,60,3312
6604,Medium,60,3312
6605,Medium,60,3312


### 3) 📈 Ranking de Progreso Histórico
**Objetivo:**  
Clasificar a los estudiantes dentro de cada nivel de `Motivation_Level` basándose en `Previous_Scores`.

In [28]:
con.execute("""
-- ¿Para qué sirve? Detectar casos atípicos de alto rendimiento
-- dentro del grupo de "Baja Motivación" para políticas de retención
SELECT
    motivation_level,
    previous_scores,
    RANK() OVER (
        PARTITION BY motivation_level
        ORDER BY Previous_Scores DESC
    ) AS rank_prev_scores
FROM student_performance
ORDER BY motivation_level, rank_prev_scores;
""").fetch_df()

,Motivation_Level,Previous_Scores,rank_prev_scores
0,High,100,1
1,High,100,1
2,High,100,1
3,High,100,1
4,High,100,1
...,...,...,...
6602,Medium,50,3322
6603,Medium,50,3322
6604,Medium,50,3322
6605,Medium,50,3322
